In [1]:
import re
from concurrent.futures import ThreadPoolExecutor

from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings


# ============================================================
# 0. CONFIG
# ============================================================

LM_STUDIO_BASE_URL = "http://127.0.0.1:1234/v1"
LM_STUDIO_MODEL = "mistralai/ministral-3-3b"   # double-check this ID matches what's loaded in LM Studio
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DOCS_DIR = "./chroma_db"
CHROMA_MEMORY_DIR = "./memory_db"
RETRIEVER_K = 3
MEMORY_K = 3
SHOW_DEBUG = True   # flip to False to hide the summary/memory debug block


# ============================================================
# 1. THINKING-TAG STRIPPER
# ============================================================
# Some models (reasoning-tuned Gemma/DeepSeek/Qwen variants served via
# LM Studio) emit their chain-of-thought wrapped in tags before the real
# answer. LangChain does NOT strip this for you - response.content contains
# everything the model streamed back. This regex removes it.

THINK_TAG_RE = re.compile(
    r"<(think|thinking|reasoning)>.*?</\1>",
    flags=re.DOTALL | re.IGNORECASE,
)


def clean(text: str) -> str:
    """Remove <think>/<thinking>/<reasoning> blocks and trim whitespace."""
    if not text:
        return ""
    return THINK_TAG_RE.sub("", text).strip()


# ============================================================
# 2. EMBEDDING MODEL (shared by both vector stores)
# ============================================================

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


# ============================================================
# 3. LLM - LM STUDIO
# ============================================================

llm = ChatOpenAI(
    model=LM_STUDIO_MODEL,
    base_url=LM_STUDIO_BASE_URL,
    api_key="lm-studio",
    temperature=0,
)


# ============================================================
# 4. VECTOR DATABASES
# ============================================================

db = Chroma(
    collection_name="games_ai",
    embedding_function=embeddings,
    persist_directory=CHROMA_DOCS_DIR,
)
retriever = db.as_retriever(search_kwargs={"k": RETRIEVER_K})

memory_db = Chroma(
    collection_name="long_term_memory",
    embedding_function=embeddings,
    persist_directory=CHROMA_MEMORY_DIR,
)

conversation_summary = ""


# ============================================================
# 5. PROMPTS / CHAINS
# ============================================================

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You rewrite the user's latest question into a standalone question.\n\n"
     "Use the conversation summary and relevant long-term memories to "
     "understand references such as: it, that, they, this, the previous one.\n\n"
     "Do NOT answer the question. Return ONLY the rewritten question, with no "
     "explanation, preamble, or reasoning."),
    ("human",
     "Conversation Summary:\n{summary}\n\n"
     "Relevant Long-Term Memories:\n{memories}\n\n"
     "Current Question:\n{question}"),
])
rewrite_chain = rewrite_prompt | llm

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful RAG assistant.\n\n"
     "Answer the user's question using the provided context. If the answer "
     "cannot be found in the context, say that you don't know. Do not invent "
     "information. Respond with only the final answer - no reasoning, no "
     "chain-of-thought, no <think> tags.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])
answer_chain = answer_prompt | llm

memory_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Identify information from this interaction that is worth remembering "
     "for future conversations.\n\n"
     "Store only useful persistent information such as: user preferences, "
     "project information, technical choices, goals, important requirements.\n\n"
     "Do NOT store: greetings, temporary questions, normal explanations, "
     "information useful only for this conversation.\n\n"
     "If there is nothing worth remembering, return: NONE\n"
     "Otherwise return each memory on a separate line. No extra commentary."),
    ("human", "User:\n{question}\n\nAssistant:\n{answer}"),
])
memory_chain = memory_prompt | llm

summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Update the conversation summary. Keep important information needed to "
     "understand future questions: current topic, important technical "
     "details, decisions, unresolved questions, important context. Keep it "
     "concise. Return ONLY the updated summary, no reasoning.\n\n"
     "Existing Summary:\n{summary}\n\n"
     "New Interaction:\nUser:\n{question}\n\nAssistant:\n{answer}"),
])
summary_chain = summary_prompt | llm


# ============================================================
# 6. MEMORY HELPERS
# ============================================================

def get_memories(query: str) -> list[str]:
    docs = memory_db.similarity_search(query, k=MEMORY_K)
    return [doc.page_content for doc in docs]


def save_memories(raw_text: str) -> None:
    if raw_text.strip().upper() == "NONE":
        return
    memories = [m.strip() for m in raw_text.split("\n") if m.strip()]
    if memories:
        memory_db.add_texts(memories)


# ============================================================
# 7. POST-PROCESSING (runs after the answer is shown)
# ============================================================
# Summary update and memory extraction are independent of each other, so
# they're run concurrently instead of one after another - this roughly
# halves the extra latency they add per turn.

def run_summary(question: str, answer: str, summary: str) -> str:
    result = summary_chain.invoke({
        "summary": summary,
        "question": question,
        "answer": answer,
    })
    return clean(result.content)


def run_memory_extraction(question: str, answer: str) -> str:
    result = memory_chain.invoke({"question": question, "answer": answer})
    return clean(result.content)


# ============================================================
# 8. MAIN CHAT LOOP
# ============================================================

def main() -> None:
    global conversation_summary

    with ThreadPoolExecutor(max_workers=2) as pool:
        while True:
            question = input("\nYou: ").strip()
            if not question:
                continue
            if question.lower() == "exit":
                break

            try:
                # --- Step 1: relevant long-term memories ---
                memories = get_memories(question)
                memory_text = "\n".join(memories)

                # --- Step 2: rewrite question (skip if no context to resolve) ---
                if conversation_summary or memory_text:
                    rewritten = rewrite_chain.invoke({
                        "summary": conversation_summary,
                        "memories": memory_text,
                        "question": question,
                    })
                    standalone_question = clean(rewritten.content) or question
                else:
                    standalone_question = question

                print("\n[Rewritten Question]")
                print(standalone_question)

                # --- Step 3 & 4: retrieve docs + build context ---
                docs = retriever.invoke(standalone_question)
                context = "\n\n".join(doc.page_content for doc in docs)

                # --- Step 5: generate answer ---
                response = answer_chain.invoke({
                    "question": standalone_question,
                    "context": context,
                })
                answer = clean(response.content)
                print("\nAI:", answer)

                # --- Steps 6 & 7: summary update + memory extraction (parallel) ---
                summary_future = pool.submit(
                    run_summary, question, answer, conversation_summary
                )
                memory_future = pool.submit(
                    run_memory_extraction, question, answer
                )

                conversation_summary = summary_future.result()
                extracted_memories = memory_future.result()

                # --- Step 8: save useful memories ---
                save_memories(extracted_memories)

                if SHOW_DEBUG:
                    print("\n-----------------------------")
                    print("CURRENT SUMMARY:")
                    print(conversation_summary)
                    print("\nMEMORIES EXTRACTED:")
                    print(extracted_memories)
                    print("-----------------------------")

            except Exception as e:
                print(f"\n[Error] Something went wrong: {e}")
                print("Is LM Studio running with a model loaded at "
                      f"{LM_STUDIO_BASE_URL}?")


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[Rewritten Question]
How Ai is used in Games

AI: AI is used in games for:
- Controlling **non-player characters (NPCs)** via finite-state machines, behavior trees, or rule-based logic
- Implementing **pathfinding** and **steering behaviors**
- Adapting **difficulty levels** dynamically
- Supporting **procedural generation** of levels, quests, dialogue, and content variations
- Enhancing **replayability** through randomized elements
- Assisting in **development workflows**, such as testing scenarios or generating assets

-----------------------------
CURRENT SUMMARY:
**Updated Summary:**
- **Current Topic:** Applications of AI in game development
- **Key Technical Details:**
  - NPC control via **finite-state machines, behavior trees, or rule-based logic**
  - Core functions: **pathfinding/steering behaviors, dynamic difficulty adjustment**
  - AI-driven **procedural generation** (levels, quests, dialogue)
- **Unresolved Questions:** Potential future expansions (e.g., advanced emotion